# Transformar Datos Deparaments
- Eliminar Registros null del compo location_id
- Eliminar registros duplicados
- Eliminar registros duplicados en base a un campo
- convertir columnas en su tipo de dato correcto
- Escribir Datos en la schema silver

In [ ]:
df_deparaments=spark.sql("SELECT * FROM zenviro.bronze.departments_py")

In [ ]:
df_departments=spark.table("zenviro.bronze.departments_py")

In [ ]:
df_departments=spark.read.table("zenviro.bronze.departments_py")

In [ ]:
display(df_departments)

In [ ]:
df_departments_not_null=df_departments.where("departments_id is not null")

In [ ]:
df_departments_not_null=df_departments.where(df_departments.departments_id.isNotNull())

In [ ]:
df_departments_not_null=df_departments.filter("departments_id is not null")

In [ ]:
df_departments_not_null=df_departments.where(df_departments.departments_id.isNotNull()).orderBy("departments_id")

In [ ]:
df_departments_distinct=df_departments_not_null.distinct()

In [ ]:
df_departments_distinct=df_departments_not_null.dropDuplicates()

In [ ]:
from pyspark.sql.functions import max

In [ ]:
df_departments_max=(
    df_departments_distinct.groupBy("departments_id")
    .agg(max("created_timestamp").alias("max_created_timestamp"))\
    .orderBy("departments_id")
    )
display(df_departments_max)

In [ ]:
department_distinct_tm=df_departments_max.join(df_departments_distinct, 
            df_departments_max.departments_id==df_departments_distinct.departments_id & 
            df_departments_max.created_timestamp==df_departments_distinct.created_timestamp,"inner")\
            .select(df_departments_distinct["*"])

display(department_distinct_tm)

In [ ]:
departments_cast_df=(
department_distinct_tm.select(department_distinct_tm.created_timestamp.cast("timestamp").alias("departments_id"),
                            department_distinct_tm.department_name,
                            department_distinct_tm.description)
)

In [ ]:
departments_cast_df.writeTo("zenviro.silver.departments_py").createOrReplace()